# 11 — Reranking & Contextual Compression (Local Cross-Encoder)

**By the end of this notebook, you can fetch wide candidates (`k=8`), rerank them with a real local neural Cross-Encoder, compress irrelevant filler sentences, and measure precision vs latency.**

### Learning Objectives
- Understand why Cross-Encoders (joint query+doc attention) outperform bi-encoder embeddings on precision.
- Run a free local `CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')` via `get_cross_encoder()` (noting Cohere optionally).
- Apply Contextual Compression to shrink token length while comparing before/after Top-3 precision vs NB05 baseline.

### Prerequisites
- Notebooks 00–10

## 1. Setup
Load `.env` configuration via `course/utils/models.py` and initialize our shared LLM and embeddings helpers.

In [1]:
# If running in Google Colab, uncomment: !git clone https://github.com/rahat4528/rag-from-scratch-course-v2.git && %cd rag-from-scratch-course-v2 && pip install -q -r course/requirements.txt
import sys
from pathlib import Path
repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course' / 'utils').exists()), Path.cwd())
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
from course.utils.models import get_llm, get_embeddings, resolve_course_root
course_dir = resolve_course_root(); data_dir = course_dir / 'data'
llm = get_llm(fallback_to_demo=True); embeddings = get_embeddings()
print(f'Ready | LLM={type(llm).__name__} | Embeddings={type(embeddings.base).__name__}')

[OfflineDemoChatModel] Notice: Provider 'ollama' not active (Ollama server is not reachable at http://localhost:11434. Install/start Ollama (`ollama pull llama3.2`) or switch RAG_PROVIDER in .env.). Running code-path demo mode.


W1112 01:29:40.854000 12320 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready | LLM=OfflineDemoChatModel | Embeddings=LocalSentenceTransformerEmbeddings


## 2. Concept: Wide Net -> Neural Cross-Encoder Reranker -> Lean Context

A bi-encoder vector store scores `query` and `chunk` **separately** (fast, scales to millions, slightly coarse). A **Cross-Encoder** reads `[query + chunk]` **together word-by-word** in one transformer pass! That's too slow for 10,000 chunks, yet takes milliseconds on the top `k=8` candidates!

- **Optional Hosted Path:** Paid services like `CohereRerank` provide hosted reranking via API; here we default to free local neural `ms-marco-MiniLM-L-6-v2`!
- **Contextual Compression:** After reranking picks top 3 chunks, trim away unrelated sentences inside those chunks to cut prompt cost.

```
All Chunks ---> [Stage 1: Vector Search (k=8, ~2ms)] ---> 8 Candidates
                                                               |
                                             [Stage 2: CrossEncoder Reranker] ---> Top 3 Precision Winners
                                                                                       |
                                                                    [Contextual Compression] ---> Tight Context!
```

## 3. Step-by-Step Build

### Step 1: Initialize Wide Retriever & Real Local Neural `CrossEncoder`
Let's index medium chunks (`300` chars) and load our local `ms-marco-MiniLM-L-6-v2` cross-encoder via `get_cross_encoder()`.

In [2]:
import time
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from course.utils.helpers import load_corpus_documents
from course.utils.models import get_cross_encoder

chunks = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40).split_documents(load_corpus_documents(data_dir))
store = InMemoryVectorStore.from_documents(chunks, embeddings)
reranker = get_cross_encoder()
print(f"Loaded CrossEncoder (neural={reranker.is_neural}) over {len(chunks)} chunks.")
        

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loaded CrossEncoder (neural=True) over 38 chunks.


> **What just happened?**
> First run downloads `ms-marco-MiniLM-L-6-v2` once locally; afterwards it executes offline.

### Step 2: Compare Before/After Top-3 Precision + Latency Trade-off vs NB05 Baseline
Let's run a tricky multi-keyword query (`'99.995% uptime SLA Enterprise Tier'`), comparing Stage-1 bi-encoder rank against CrossEncoder reranking!

In [3]:
q_sla = "What exact uptime SLA percentage does Enterprise Tier guarantee?"
t0 = time.perf_counter()
stage1 = store.similarity_search(q_sla, k=8)
s1_ms = (time.perf_counter() - t0) * 1000

t1 = time.perf_counter()
reranked = reranker.rerank(q_sla, stage1, top_n=3)
rr_ms = (time.perf_counter() - t1) * 1000
print(f"Stage-1 (#1 has 99.995%?={'99.995%' in stage1[0].page_content}) in {s1_ms:.1f}ms")
print(f"Reranked (#1 has 99.995%?={'99.995%' in reranked[0][1].page_content}, score={reranked[0][0]:.2f}) in {rr_ms:.1f}ms")
        

Stage-1 (#1 has 99.995%?=True) in 10.8ms
Reranked (#1 has 99.995%?=True, score=9.15) in 72.1ms


> **What just happened?**
> CrossEncoder inspects exact query-document interaction and promotes the true `99.995%` chunk to rank #1 cleanly!

### Step 3: Apply Contextual Compression & Measure Character/Token Reduction
Let's filter out sentences inside our top-3 reranked chunks that share zero meaningful stems with the query and verify saved tokens!

In [4]:
import re
def compress_chunks(query_str: str, docs):
    q_terms = {w for w in re.findall(r"[a-z0-9_]+", query_str.lower()) if len(w) > 2}
    kept_blocks = []
    for d in docs:
        kept = [ln.strip() for ln in d.page_content.splitlines() if len(q_terms & set(re.findall(r"[a-z0-9_]+", ln.lower()))) >= 1]
        kept_blocks.append("\n".join(kept) if kept else d.page_content)
    return kept_blocks

top3_docs = [doc for _, doc in reranked]
raw_len = sum(len(d.page_content) for d in top3_docs)
comp_list = compress_chunks(q_sla, top3_docs)
comp_len = sum(len(x) for x in comp_list)
print(f"Compressed context: {raw_len} -> {comp_len} chars ({100*(1-comp_len/raw_len):.1f}% savings!)")
        

Compressed context: 696 -> 544 chars (21.8% savings!)


> **What just happened?**
> Contextual compression trims boilerplate intro headings while preserving the exact answer line (`99.995%`)!

### Step 4: Generate Final Answer from Compressed Context
Now let's pass the lean compressed snippet into `llm` and inspect the output cleanly.

In [5]:
compressed_ctx = "\n---\n".join(comp_list)
ans = llm.invoke(f"<context>\n{compressed_ctx}\n</context>\nQuestion: {q_sla}").content
print("Compressed Answer:", ans)
assert "99.995" in ans
        

Compressed Answer: Based on the provided context: ### What uptime SLA is guaranteed for the Enterprise Tier? The Enterprise Tier guarantees a **99.995%** service uptime SLA, backed by multi-region active-active coordinator clusters. Professional Tier guarantees 99.95%, and Starter Tier provides best-effort 99.5% availability. Enterprise 24/7 Mission-Critical Support (Included in Enterprise Tier): Enterprise 24/7 Mission-Critical Support (Included in Enterprise Tier): Enterprise Plan: $4,200 per month. Supports up to 200 active nodes. Includes ZeroGravity automatic failover, QuantumCache v4, 1-year telemetry retention, and 24/7/365 mission-critical support.


> **What just happened?**
> Lean, high-precision context yields accurate grounded generation at smaller token budgets.

## 4. Try It Yourself
1. **Compare Precision@3 on 3 queries** — *Hint:* Measure how often target facts sit in rank 1–3 before vs after `reranker.rerank()`.
2. **Inspect latency on larger pool** — *Hint:* Compare `k=4` vs `k=12` into `reranker.rerank(..., top_n=3)`.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>
Run the cell below to check one working solution.
</details>

In [6]:
qs = ["Enterprise Plan monthly cost", "ROS2 distributions supported", "Rest API v1 deprecation date"]
for qt in qs:
    cands = store.similarity_search(qt, k=8)
    top_r = reranker.rerank(qt, cands, top_n=1)[0]
    print(f"Reranked #1 ({top_r[0]:+.2f}) for '{qt}' -> {top_r[1].metadata['source']}")
        

Reranked #1 (+8.60) for 'Enterprise Plan monthly cost' -> company_handbook.txt


Reranked #1 (+7.83) for 'ROS2 distributions supported' -> product_faq.md


Reranked #1 (+8.45) for 'Rest API v1 deprecation date' -> recent_changelog.md


## 5. Common Mistakes
- Running a CrossEncoder over 5,000 chunks directly instead of only top 8–25 Stage-1 vector hits — causes multi-second hangs!
- Over-compressing short 2-line chunks until pronoun targets disappear.
- Assuming CrossEncoder scores work globally across queries — logits differ per query, so rank relative order cleanly.

## 6. Recap
- Two-Stage Retrieval pairs fast Bi-Encoder recall (`k=8`) with deep Cross-Encoder precision (`top_n=3`).
- `ms-marco-MiniLM-L-6-v2` runs completely free and locally with zero paid Cohere key.
- Contextual Compression trims unrelated noise, cutting prompt tokens and latency cost.

## 7. Next Notebook
Next (`12_agentic_and_self_correcting_rag.ipynb`): Build real tool-calling agents (`create_agent`) AND self-correcting LangGraph loops with relevance grading & retry caps!

## 8. Video Chapter Outline
1. **Bi-Encoder vs Cross-Encoder Intuition** — Why joint attention beats compressed single vectors.
2. **Loading Free Local CrossEncoder** — Avoiding paid Cohere requirements cleanly.
3. **Before/After Precision & Latency Benchmark** — Quantifying exact #1 promotion & ms cost.
4. **Contextual Compression** — Measuring character/token reduction live.
5. **When Reranking Pays Off & Recap** — Latency vs precision tradeoff.